Issues and achievements in our setup:


*   Clause type leaked in instruction prompt, should have been only in response prompt. Model would have learnt a pattern, where it sees the clause type in instruction and the same in response. It would have learnt to copy the clause type IF it is found in instruction, ignores to generate it otherwise. So, our model DID NOT learn clause type classificaion. I dont wanna retrain for 6 more hours without the clause_type in the instruction prompt.
*   There is a problem with the loss function we used. Our cross entropy loss compares the predicted and the GT tokens. This wont work because, our task is a mix of classification and summarization. If our model predicts Low risk but GT is high risk, and the summary is reasonably similar, loss would be less but still we get the wrong classification. We would need 2 different loss functions or may be more importance to the critical tokens.
* Our model however, is good at legal text summarization and providing a justification for why it chose a particular risk level.
*No hallucination like the original 3b model, co-herent and meaningful sentences seen.



In [1]:
!pip cache purge
!pip install -Uq accelerate trl peft datasets
!pip uninstall -y transformers
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -Uq bitsandbytes
!pip install -q groq
!pip install -q huggingface_hub


Files removed: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.8 MB/s eta 0:00:00
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 13.3 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq

api_key = "my_api_key"
os.environ["GROQ_API_KEY"] = api_key

client = Groq(api_key = api_key)

try:
    test_response = client.chat.completions.create(model = "llama-3.3-70b-versatile",messages=[{"role": "user", "content": "Respond with: Lets get rollin! API working!"}],
                                                 max_tokens = 20, temperature = 0.1)

    print(f"Response: {test_response.choices[0].message.content}")
    print("\n Groq API is ready!")
    print(f" Model: Llama 3.3 70B")

except Exception as e:
    print(f"Error: {e}")
    print("Check your API key and try again.")

Response: Lets get rollin! API working!

 Groq API is ready!
 Model: Llama 3.3 70B


In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
df = pd.read_csv('cuad_enriched.csv')

print(f"columns: {df.columns.to_list()}")
print(f"contracts: {df['contract_name'].nunique()}")
print(f"clause types {df['clause_type'].nunique()}")

Saving cuad_enriched.csv to cuad_enriched.csv
columns: ['contract_name', 'clause_type', 'clause_text', 'enriched_context', 'metadata']
contracts: 510
clause types 41


In [ ]:
df['enriched_context'].iloc[11]

'**Contract Context:**\n    Contract Type: DISTRIBUTOR  AGREEMENT\nJurisdiction : Illinois\nRenewal Term: 10 1 years\nNotice Period to terminate renewal: None\nKey Provisions: Change of Control,Cap on Liability,Minimum Commitment\n\n    **Target Clause Type:** Governing Law\n    **Clause Text:**\n    this agreement is to be construed according to the laws of the state of illinois.'

In [ ]:

#Prompt for the 70B model.

def create_label_generation_prompt(enriched_context):
    """
    Creates a structured prompt for the 70B model

    Why this structure:
    - System prompt: Sets the role and output format ONCE
    - User prompt: The actual clause to analyze
    - JSON format: Easy to parse programmatically
    """

    system_prompt = """You are an expert legal contract analyst with 20 years of experience reviewing commercial contracts.

Your task is to analyze contract clauses and provide:
1. Risk assessment (HIGH, MEDIUM, or LOW)
2. Brief explanation of the risk
3. Plain English summary

CRITICAL RULES:
- Respond ONLY in valid JSON format
- No text before or after the JSON
- Risk level MUST be exactly: HIGH, MEDIUM, or LOW
- Consider the full contract context provided
- Base risk on jurisdiction, Renewal conditions, contract type and especially on the clause content

Response format:
{
    "risk_level": "HIGH/MEDIUM/LOW",
    "risk_reason": "2-3 sentences explaining the risk considering contract context",
    "plain_summary": "1-2 sentences in simple everyday language"
}"""

    user_prompt = f"""Analyze this contract clause:

{enriched_context}

Remember: Respond ONLY with valid JSON, nothing else."""

    return system_prompt, user_prompt

# Test it on one example
sample = df['enriched_context'].iloc[10]

system_prompt, user_prompt = create_label_generation_prompt(sample)

print("=== SYSTEM PROMPT ===")
print(system_prompt)
print("\n=== USER PROMPT ===")
print(user_prompt)

=== SYSTEM PROMPT ===
You are an expert legal contract analyst with 20 years of experience reviewing commercial contracts.

Your task is to analyze contract clauses and provide:
1. Risk assessment (HIGH, MEDIUM, or LOW)
2. Brief explanation of the risk
3. Plain English summary

CRITICAL RULES:
- Respond ONLY in valid JSON format
- No text before or after the JSON
- Risk level MUST be exactly: HIGH, MEDIUM, or LOW
- Consider the full contract context provided
- Base risk on jurisdiction, Renewal conditions, contract type and especially on the clause content

Response format:
{
    "risk_level": "HIGH/MEDIUM/LOW",
    "risk_reason": "2-3 sentences explaining the risk considering contract context",
    "plain_summary": "1-2 sentences in simple everyday language"
}

=== USER PROMPT ===
Analyze this contract clause:

**Contract Context:**
    Contract Type: DISTRIBUTOR  AGREEMENT
Jurisdiction : Illinois
Renewal Term: 10 1 years
Notice Period to terminate renewal: None
Key Provisions: Change

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/CUAD_Project'
os.makedirs(DRIVE_PATH, exist_ok=True)

Mounted at /content/drive


In [ ]:
#removing clause_types that are present as metadata:

SKIP_CLAUSE_TYPES = [
    'Document Name',
    'Parties',
    'Agreement Date',
    'Effective Date',
    'Expiration Date',
]

df_to_label = df[~(df['clause_type'].isin(SKIP_CLAUSE_TYPES))].reset_index(drop=True)

sampled_df = df_to_label.groupby('clause_type').apply(lambda x: x.sample(min(110, len(x)), random_state=42)).reset_index(drop=True)
sampled_df.to_csv('sampled_df.csv', index=False)
sampled_df.shape

/tmp/ipython-input-3969280879.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df_to_label.groupby('clause_type').apply(lambda x: x.sample(min(110, len(x)), random_state=42)).reset_index(drop=True)


(3455, 5)

In [ ]:
#creating a new sample from the remaining
remaining = pd.concat([df, sampled_df]).drop_duplicates(subset = ['enriched_context'], keep= False)
sampled_new = remaining.groupby('clause_type').apply(lambda x: x.sample(min(110, len(x)), random_state = 42)).reset_index(drop=True)
sampled_new.shape

/tmp/ipython-input-965187492.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_new = remaining.groupby('clause_type').apply(lambda x: x.sample(min(110, len(x)), random_state = 42)).reset_index(drop=True)


(2749, 5)

In [ ]:
#NOTE: REFER DEFINITON OF "re_run_required" FIELD IN THE BELOW CELLS.

In [ ]:
import json
from tqdm import tqdm
import time
import pandas as pd

CHECKPOINT_FILE = f'{DRIVE_PATH}/labeled_clauses.csv'
CHECKPOINT_EVERY = 50

def generate_label(enriched_context, client, max_retries=3):

  system_prompt, user_prompt = create_label_generation_prompt(enriched_context)

  for attempt in range(max_retries):
    try:
      response = client.chat.completions.create(model = "llama-3.3-70b-versatile", messages = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_prompt}],
                                     max_tokens = 300, temperature =0.1)
      response_text = response.choices[0].message.content.strip()

      parsed = json.loads(response_text)

      assert "risk_level" in parsed
      assert "risk_reason" in parsed
      assert "plain_summary" in parsed
      assert parsed["risk_level"] in ["HIGH", "MEDIUM", "LOW"]

      return parsed

    except json.JSONDecodeError:
      print(f"JSONDecodeError on attempt {attempt + 1}. Retrying...")
      time.sleep(3)

    except AssertionError:
      print(f"AssertionError[Invalid format returned] on attempt {attempt + 1}. Retrying...")
      time.sleep(3)

    except Exception as e:
      print(f"Error on attempt {attempt + 1}: {e}. Retrying...")
      time.sleep(3)

  return {
        "risk_level": "MEDIUM",
        "risk_reason": "Could not generate label after retries",
        "plain_summary": "Could not generate summary after retries"
    }


def run_label_generation(sampled_df, client):
    """
    Main generation loop with checkpointing
    """
    # Check for existing checkpoint
    if os.path.exists(CHECKPOINT_FILE):
        print(f"Checkpoint found: {CHECKPOINT_FILE}")
        labeled_df = pd.read_csv(CHECKPOINT_FILE)
        start_idx = len(labeled_df)
        print(f" Checkpoint found! Resuming from clause {start_idx}")
    else:
        labeled_df = pd.DataFrame()
        start_idx = 0
        print(f" Starting fresh generation")

    print(f"Total clauses: {len(sampled_df)}")#changing this to re-run the failed samples.
    print(f"Remaining: {2749+3455 - start_idx}")#len(sampled_df)
    print(f"Rate limit: ~30 requests/min, 1000/day\n")


    results = []

    for idx in tqdm(range(start_idx, 3455+2760)):#changing end index to 787 from 3455
      row = sampled_df.iloc[idx-3455]

      # Generate label
      label = generate_label(row['enriched_context'], client)

      # Combine with original data
      results.append({
          'contract_name': row['contract_name'],
          'clause_type': row['clause_type'],
          'clause_text': row['clause_text'],
          'enriched_context': row['enriched_context'],
          'risk_level': label['risk_level'],
          'risk_reason': label['risk_reason'],
          'plain_summary': label['plain_summary']
      })

      if (idx + 1) % CHECKPOINT_EVERY == 0:
          checkpoint_df = pd.concat(
              [labeled_df, pd.DataFrame(results)],
              ignore_index=True
          )

          checkpoint_df.to_csv(CHECKPOINT_FILE, index=False)
          print(f"\n Checkpoint saved: {idx + 1} clauses labeled")

      time.sleep(2)
      final_df = pd.concat(
        [labeled_df, pd.DataFrame(results)],
        ignore_index=True
      )
      final_df.to_csv(CHECKPOINT_FILE, index=False)

      print(f"\n Generation complete!")
      print(f" Total labeled: {len(final_df)} clauses")

    return final_df


labelled_df_new = run_label_generation(sampled_new, client)




Checkpoint found: /content/drive/MyDrive/CUAD_Project/labeled_clauses.csv
 Checkpoint found! Resuming from clause 4600
Total clauses: 2749
Remaining: 1604
Rate limit: ~30 requests/min, 1000/day



  0%|          | 1/1615 [00:03<1:22:33,  3.07s/it]


 Generation complete!
 Total labeled: 4601 clauses


  0%|          | 2/1615 [00:06<1:24:55,  3.16s/it]


 Generation complete!
 Total labeled: 4602 clauses


...


 Generation complete!
 Total labeled: 4768 clauses
Error on attempt 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khry8nsje83r1djj1f1fqqrn` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99765, Requested 344. Please try again in 1m34.176s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying...
Error on attempt 2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khry8nsje83r1djj1f1fqqrn` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99761, Requested 344. Please try again in 1m30.72s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying...
Error on attempt 3: Error code: 429 - {'error': {'message': 'Rate l

 10%|█         | 169/1615 [11:30<4:39:21, 11.59s/it]


 Generation complete!
 Total labeled: 4769 clauses
Error on attempt 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khry8nsje83r1djj1f1fqqrn` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99751, Requested 330. Please try again in 1m9.984s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying...
Error on attempt 2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khry8nsje83r1djj1f1fqqrn` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99748, Requested 330. Please try again in 1m7.391999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying...
Error on attempt 3: Error code: 429 - {'error': {'message': 'R

 10%|█         | 169/1615 [11:41<1:40:04,  4.15s/it]


KeyboardInterrupt: 

In [ ]:
CHECKPOINT_FILE = f'{DRIVE_PATH}/labeled_clauses.csv'
labelled_df = pd.read_csv(CHECKPOINT_FILE)

In [ ]:
labelled_df.shape

(4751, 7)

In [ ]:
labelled_df = labelled_df[~(labelled_df['risk_reason'].str.lower().str.contains('could not generate'))]
labelled_df.to_csv(CHECKPOINT_FILE, index = False)

In [ ]:
labelled_df.shape

(4751, 7)

In [ ]:
labelled_df.shape

(4751, 7)

In [ ]:
#taking the samples that need to be re-run, cos the summary was not generated.
re_run_required = labelled_df[labelled_df['plain_summary'].str.lower().str.contains('could not generate')]
re_run_required.shape

(0, 7)

In [ ]:
labelled_df_new=labelled_df[~(labelled_df['plain_summary'].str.lower().str.contains('could not generate'))]
labelled_df_new.shape

(3439, 7)

In [ ]:
labelled_df_new.to_csv(CHECKPOINT_FILE, index=False)

In [ ]:
remaining = pd.concat([df, sampled_df]).drop_duplicates(keep=False)
remaining.shape

(10361, 5)

In [ ]:
excel = pd.read_excel(f'{DRIVE_PATH}/labeled_clauses_proper.xlsx')
excel.shape

(3452, 7)

In [ ]:
labelled_df.shape

(4751, 7)

In [ ]:
combined_df = pd.concat([labelled_df, excel], ignore_index=True).drop_duplicates(subset = ['enriched_context'], keep='first')
combined_df.shape

(5488, 8)

In [ ]:
#reading the combined_csv from drive
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

import os
DRIVE_PATH = '/content/drive/MyDrive/CUAD_Project'
combined_df = pd.read_csv(f'{DRIVE_PATH}/final_clauses.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def format_instruction_response(row):
  instruction = f"""Analyse the following clause and provide:
1. Clause type classification
2. Risk assessment (HIGH/MEDIUM/LOW) with explanation
3. Plain English summary

{row['enriched_context']}

Provide your analysis:

"""
  response = f"""**Clause Type:** {row['clause_type']}

**Risk Assessment:** {row['risk_level']}
**Explanation:** {row['risk_reason']}

**Plain English Summary:** {row['plain_summary']}"""

  full_text = f"{instruction}\n\n{response}"

  return {'instruction': instruction, 'response': response, 'full_text': full_text}



training_data = combined_df.apply(format_instruction_response, axis=1).tolist()
training_df = pd.DataFrame(training_data)
training_df.head(2)

print(f"Created {len(training_df)} training examples")

print(f"\nTraining Example:")
print("="*80)
print("INSTRUCTION:")
print(training_df['instruction'].iloc[0][:300] + "...")
print("\nRESPONSE:")
print(training_df['response'].iloc[0][:300] + "...")
print("="*80)



Created 5488 training examples

Training Example:
INSTRUCTION:
Analyse the following clause and provide:
1. Clause type classification
2. Risk assessment (HIGH/MEDIUM/LOW) with explanation
3. Plain English summary

**Contract Context:**
    Contract Type: INTELLECTUAL PROPERTY AGREEMENT
Jurisdiction : Georgia
Renewal Term: Not Applicable
Notice Period to termin...

RESPONSE:
**Clause Type:** Affiliate License-Licensee

**Risk Assessment:** MEDIUM
**Explanation:** The sublicense rights granted to Equifax may pose a risk of losing control over the licensed Certegy materials, as Equifax is allowed to grant sublicenses to other parties. This could lead to potential intellec...


In [ ]:
print(training_df.iloc[0]['full_text'])

Analyse the following clause and provide:
1. Clause type classification
2. Risk assessment (HIGH/MEDIUM/LOW) with explanation
3. Plain English summary

**Contract Context:**
    Contract Type: INTELLECTUAL PROPERTY AGREEMENT
Jurisdiction : Georgia
Renewal Term: Not Applicable
Notice Period to terminate renewal: None
Key Provisions: Change of Control,Cap on Liability,Minimum Commitment

    **Target Clause Type:** Affiliate License-Licensee
    **Clause Text:**
    the sublicense rights granted to equifax pursuant to section 4.4(a) include the right for equifax to grant sublicenses to the licensed certegy materials (excluding the utility

Provide your analysis:



**Clause Type:** Affiliate License-Licensee

**Risk Assessment:** MEDIUM
**Explanation:** The sublicense rights granted to Equifax may pose a risk of losing control over the licensed Certegy materials, as Equifax is allowed to grant sublicenses to other parties. This could lead to potential intellectual property infringement o

In [ ]:
#TRAIN/VAL/TEST SPLIT

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(training_df, test_size=0.2, random_state=42, stratify = combined_df['clause_type'])
train_df.to_csv(f'{DRIVE_PATH}/train_new.csv', index=False)
val_df.to_csv(f'{DRIVE_PATH}/val_new.csv', index=False)

print(f"Train: {len(train_df)}")
print(f"Val: {len(val_df)}")


Train: 4390
Val: 1098


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

import os
DRIVE_PATH = '/content/drive/MyDrive/CUAD_Project'

train_df = pd.read_csv(f'{DRIVE_PATH}/train_new.csv')
val_df = pd.read_csv(f'{DRIVE_PATH}/val_new.csv')

Mounted at /content/drive


In [ ]:
train_df.columns

Index(['instruction', 'response', 'full_text'], dtype='object')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct" #the model we need to finetune

#initialising the quantisation config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.float16
)

#initialising the base model, passing the quantisation config to it.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config = bnb_config, device_map = 'auto', trust_remote_code = True, )


#loading the right tokenizer, autotokenizer takes care of it.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


#prepare the model for k-bit training, this ensures gradient checkpointing, makes sure the base model weights are frozen
#and carries out the computation in float32 for layernorm and output layers, to ensure precision in calculation.
model = prepare_model_for_kbit_training(model)
print("successfully loaded base model in 4 bit")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

successfully loaded base model in 4 bit
Memory used: 3.04 GB


In [ ]:
#LoRA configuration

lora_config = LoraConfig(r=16,
                         lora_alpha=32,
                         target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                           'up_proj', 'down_proj', 'gate_proj'],#up, down and gate proj correspond to the layers in the mlp
                         lora_dropout=0.05,
                         bias='none',
                         task_type='CAUSAL_LM')

#Apply LoRA to model
model = get_peft_model(model, lora_config)
print("successfully applied LoRA")


model.print_trainable_parameters()



successfully applied LoRA
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [ ]:
# ========================================
# DIAGNOSTIC: Check Your Actual Lengths
# ========================================

# Check your formatted data lengths
print(" Checking sequence lengths in your data...\n")

sample_lengths = []

for i in range(min(100, len(train_df))):
    instruction = train_df.iloc[i]['instruction']
    response = train_df.iloc[i]['response']

    # Format as chat
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": response}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize to get length
    tokens = tokenizer(text, add_special_tokens=True)
    length = len(tokens['input_ids'])
    sample_lengths.append(length)

# Statistics
import numpy as np
print(f"Sequence Length Statistics (100 samples):")
print(f"   Min:    {min(sample_lengths)}")
print(f"   Max:    {max(sample_lengths)}")
print(f"   Mean:   {np.mean(sample_lengths):.0f}")
print(f"   Median: {np.median(sample_lengths):.0f}")
print(f"   95th percentile: {np.percentile(sample_lengths, 95):.0f}")

# How many exceed 2048?
over_limit = sum(1 for l in sample_lengths if l > 2048)
print(f"\n Sequences over 2048 tokens: {over_limit}/{len(sample_lengths)} ({over_limit/len(sample_lengths)*100:.1f}%)")

if over_limit > 0:
    print(f"   These will be TRUNCATED, losing response data!")



 Checking sequence lengths in your data...

Sequence Length Statistics (100 samples):
   Min:    261
   Max:    577
   Mean:   356
   Median: 338
   95th percentile: 481

 Sequences over 2048 tokens: 0/100 (0.0%)


In [ ]:
from datasets import Dataset

def format_for_training(examples):

  formatted_texts = []

  for instruction, response in zip(examples['instruction'], examples['response']):

    messages = [{'role': 'user', 'content': instruction}, {'role': 'assistant', 'content': response}]

    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)

    formatted_texts.append(text)


  #making return_attention_mask as True cos the datacollator will use the mask as a template to pad... Dynamic padding will be taken care of by the datacollator.
  model_inputs = tokenizer(formatted_texts, padding = False, max_length = 2048, truncation = True, return_tensors = None, return_attention_mask = True)

  #no need to pad here.
  model_inputs["labels"] = [input_ids.copy() for input_ids in model_inputs["input_ids"]]

  labels =[]

  #Instruction Masking...[probably not drastic immprovement in scores, still a good learning exercise]
  for i,(instruction, response) in enumerate(zip(examples['instruction'], examples['response'])):

    response_tokens = tokenizer(response, add_special_tokens = False, max_length = 2048, truncation = True)['input_ids']

    full_text = model_inputs['input_ids'][i]

    response_start = len(full_text) - len(response_tokens)

    temp = [-100] *len(full_text) #start with all tokens masked

    temp[response_start:] = full_text[response_start:] #unmask only the response tokens

    labels.append(temp)

  model_inputs['labels'] = labels

  return model_inputs




print("Converting data to HF Dataset format...")

train_dataset = Dataset.from_pandas(train_df[['instruction', 'response']])
val_dataset = Dataset.from_pandas(val_df[['instruction', 'response']])

print("Tokenizing Datasets...")

train_dataset = train_dataset.map(format_for_training, batched = True, remove_columns = ['instruction', 'response'], desc = "Formatting Train data")
val_dataset = val_dataset.map(format_for_training, batched = True, remove_columns = ['instruction', 'response'], desc = "Formatting Val data")

print("Datasets tokenized and formatted!")

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")


print(f"\n Sample verification:")
for i in range(3):
    labels = train_dataset[i]['labels']
    learnable = sum(1 for l in labels if l != -100)
    print(f"Sample {i}: {learnable} learnable tokens")



Converting data to HF Dataset format...
Tokenizing Datasets...


Formatting Train data:   0%|          | 0/4390 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

In [ ]:
#Training Hyperparameters:

from transformers import TrainingArguments

OUTPUT_DIR = f"{DRIVE_PATH}/llama-3.2-3b-contract-analyzer"

os.environ["TENSORBOARD_LOGGING_DIR"] = f"{DRIVE_PATH}/logs2"

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    run_name = 'contract-clause-analysis',

    #Training Schedule
    num_train_epochs = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 16,
    per_device_eval_batch_size = 1,

    #Learning Rate params

    learning_rate = 2e-4,
    lr_scheduler_type = 'cosine',
    warmup_ratio = 0.03, #LR starts from 0 and reaches the peak of 2e-4 at the 3% of the total training steps and declines from there, big updates early on, finegrained updates towards the end.

    #Optimization
    optim = 'adamw_torch', #The paged version pushes data to RAM if VRAM is about to run out of memory, 8 bit version optimizes memory footprint.
    weight_decay = 0.01, #Adds a penalty to the model weights itself, not the LR, to make sure no weight is over-focused.
    max_grad_norm = 1.0, #Clips gradient and does not allow a norm of more than 1


    #Evaluation and Saving
    eval_strategy = 'steps', #model run on eval data every 50 train steps
    eval_steps = 100,
    save_strategy = 'steps', #model weights saved every 50 steps, usually in sync with the eval strategy so that we dont lose any info that was used for eval
    save_steps = 100,
    save_total_limit = 3, #saves the last 3 checkpoints.
    load_best_model_at_end = True,
    metric_for_best_model = 'eval_loss',

    #Logging
    logging_dir = f"{DRIVE_PATH}/logs",
    logging_steps = 10,
    report_to = 'none',

    #Memory Optimization
    gradient_checkpointing = True,
    fp16 =False,
    bf16 = False,

    remove_unused_columns = False,
    label_names = ['labels']

)

print("Training arguments configured")
print(f"\nKey Settings:")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total epochs: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Warmup steps: ~{int(len(train_dataset) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs * training_args.warmup_ratio)}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments configured

Key Settings:
Effective batch size: 16
Total epochs: 3
Learning rate: 0.0002
Warmup steps: ~24


In [ ]:
#Supervised Fine-Tuning

from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling
from transformers import DataCollatorForSeq2Seq


#data_collator = DataCollatorForLanguageModeling(tokenizer, mlm = False)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,        # Pad dynamically
    pad_to_multiple_of=8,  # For efficiency
    label_pad_token_id=-100,  # Pad labels with -100
    return_tensors="pt"
)

#passing the datacollator handles the dynamic padding, max length sequence per batch

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    args = training_args,
    processing_class = tokenizer,
    eval_dataset = val_dataset,
    data_collator = data_collator
)

print("Trainer initialized!")
print("\nTraining will begin with:")
print(f"Total training steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'calculating...'}")
print(f"Evaluation every: {training_args.eval_steps} steps")
print(f"Checkpoint every: {training_args.save_steps} steps")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Truncating train dataset:   0%|          | 0/4390 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1098 [00:00<?, ? examples/s]

Trainer initialized!

Training will begin with:
Total training steps: 0
Evaluation every: 100 steps
Checkpoint every: 100 steps


In [ ]:


import time

print("STARTING FINE-TUNING!")
print("Estimated time: 4-6 hours")
print("Checkpoints: Every 100 steps to Google Drive\n")

start_time = time.time()

# Train (has built-in progress bar)
trainer.train()

# Done
elapsed = (time.time() - start_time) / 3600
print(f"\nTRAINING COMPLETE in {elapsed:.2f} hours")

# Save
print("Saving model...")
final_model_path = f"{DRIVE_PATH}/final_model"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Saved to: {final_model_path}")

STARTING FINE-TUNING!
Estimated time: 4-6 hours
Checkpoints: Every 100 steps to Google Drive



Step,Training Loss,Validation Loss
100,0.649255,0.663464
200,0.676968,0.690782
300,0.595724,0.656336
400,0.587755,0.661930
500,0.560337,0.635147
600,0.468333,0.636861
700,0.452866,0.628698


**Reloading Checkpoint beccause we lost GPU access **

In [ ]:
#Re-Install the libraries and load the data files before running this cell...
import os

checkpoint_dir = f"{DRIVE_PATH}/llama-3.2-3b-contract-analyzer"
latest_checkpoint = f"{checkpoint_dir}/checkpoint-700"

print(f"Checkpoint loaded : {latest_checkpoint}")

Checkpoint loaded : /content/drive/MyDrive/CUAD_Project/llama-3.2-3b-contract-analyzer/checkpoint-700


In [ ]:
#Pasting the model loading cell here, making some changes to load from checkpoint:

#NOTE: Had to login to HF space with a token from account, as we are running from a different co

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import torch

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct" #the model we need to finetune

#initialising the quantisation config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.float16
)

#initialising the base model, passing the quantisation config to it.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config = bnb_config, device_map = 'auto', trust_remote_code = True)

#Prepare base model for kbit training
base_model = prepare_model_for_kbit_training(model)

#apply the cehckpointed lora adapters to the base model
model = PeftModel.from_pretrained(base_model, latest_checkpoint, is_trainable = True, torch_dtype=torch.float16)


#loading the right tokenizer, autotokenizer takes care of it.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'




#loading model from latest checkpoint
print(f"successfully loaded base model from checkpoint - {latest_checkpoint}")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

successfully loaded base model from checkpoint - /content/drive/MyDrive/CUAD_Project/llama-3.2-3b-contract-analyzer/checkpoint-700
Memory used: 3.13 GB


In [ ]:
#Tokenize the data as done before


from datasets import Dataset

def format_for_training(examples):

  formatted_texts = []

  for instruction, response in zip(examples['instruction'], examples['response']):

    messages = [{'role': 'user', 'content': instruction}, {'role': 'assistant', 'content': response}]

    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)

    formatted_texts.append(text)


  #making return_attention_mask as True cos the datacollator will use the mask as a template to pad... Dynamic padding will be taken care of by the datacollator.
  model_inputs = tokenizer(formatted_texts, padding = False, max_length = 2048, truncation = True, return_tensors = None, return_attention_mask = True)

  #no need to pad here.
  model_inputs["labels"] = [input_ids.copy() for input_ids in model_inputs["input_ids"]]

  labels =[]

  #Instruction Masking...[probably not drastic immprovement in scores, still a good learning exercise]
  for i,(instruction, response) in enumerate(zip(examples['instruction'], examples['response'])):

    response_tokens = tokenizer(response, add_special_tokens = False, max_length = 2048, truncation = True)['input_ids']

    full_text = model_inputs['input_ids'][i]

    response_start = len(full_text) - len(response_tokens)

    temp = [-100] *len(full_text) #start with all tokens masked

    temp[response_start:] = full_text[response_start:] #unmask only the response tokens

    labels.append(temp)

  model_inputs['labels'] = labels

  return model_inputs




print("Converting data to HF Dataset format...")

train_dataset = Dataset.from_pandas(train_df[['instruction', 'response']])
val_dataset = Dataset.from_pandas(val_df[['instruction', 'response']])

print("Tokenizing Datasets...")

train_dataset = train_dataset.map(format_for_training, batched = True, remove_columns = ['instruction', 'response'], desc = "Formatting Train data")
val_dataset = val_dataset.map(format_for_training, batched = True, remove_columns = ['instruction', 'response'], desc = "Formatting Val data")

print("Datasets tokenized and formatted!")

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")


print(f"\n Sample verification:")
for i in range(3):
    labels = train_dataset[i]['labels']
    learnable = sum(1 for l in labels if l != -100)
    print(f"Sample {i}: {learnable} learnable tokens")



Converting data to HF Dataset format...
Tokenizing Datasets...


Formatting Train data:   0%|          | 0/4390 [00:00<?, ? examples/s]

Formatting Val data:   0%|          | 0/1098 [00:00<?, ? examples/s]

Datasets tokenized and formatted!
Train dataset: 4390 examples
Val dataset: 1098 examples

 Sample verification:
Sample 0: 154 learnable tokens
Sample 1: 121 learnable tokens
Sample 2: 114 learnable tokens


In [ ]:
#Paste all the training args and training init and re-run

from transformers import TrainingArguments

OUTPUT_DIR = f"{DRIVE_PATH}/llama-3.2-3b-contract-analyzer"

os.environ["TENSORBOARD_LOGGING_DIR"] = f"{DRIVE_PATH}/logs2"

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    run_name = 'contract-clause-analysis',

    #Training Schedule
    num_train_epochs = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 16,
    per_device_eval_batch_size = 1,

    #Learning Rate params

    learning_rate = 2e-4,
    lr_scheduler_type = 'cosine',
    warmup_ratio = 0.03, #LR starts from 0 and reaches the peak of 2e-4 at the 3% of the total training steps and declines from there, big updates early on, finegrained updates towards the end.

    #Optimization
    optim = 'adamw_torch', #The paged version pushes data to RAM if VRAM is about to run out of memory, 8 bit version optimizes memory footprint.
    weight_decay = 0.01, #Adds a penalty to the model weights itself, not the LR, to make sure no weight is over-focused.
    max_grad_norm = 1.0, #Clips gradient and does not allow a norm of more than 1


    #Evaluation and Saving
    eval_strategy = 'steps', #model run on eval data every 50 train steps
    eval_steps = 100,
    save_strategy = 'steps', #model weights saved every 50 steps, usually in sync with the eval strategy so that we dont lose any info that was used for eval
    save_steps = 100,
    save_total_limit = 3, #saves the last 3 checkpoints.
    load_best_model_at_end = True,
    metric_for_best_model = 'eval_loss',

    #Logging
    logging_dir = f"{DRIVE_PATH}/logs",
    logging_steps = 10,
    report_to = 'none',

    #Memory Optimization
    gradient_checkpointing = True,
    fp16 =False,
    bf16 = False,

    remove_unused_columns = False,
    label_names = ['labels']

)

print("Training arguments configured")
print(f"\nKey Settings:")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total epochs: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Warmup steps: ~{int(len(train_dataset) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs * training_args.warmup_ratio)}")


#Supervised Fine-Tuning

from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling
from transformers import DataCollatorForSeq2Seq


#data_collator = DataCollatorForLanguageModeling(tokenizer, mlm = False)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,        # Pad dynamically
    pad_to_multiple_of=8,  # For efficiency
    label_pad_token_id=-100,  # Pad labels with -100
    return_tensors="pt"
)

#passing the datacollator handles the dynamic padding, max length sequence per batch

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    args = training_args,
    processing_class = tokenizer,
    eval_dataset = val_dataset,
    data_collator = data_collator
)

print("Trainer initialized!")
print("\nTraining will begin with:")
print(f"Total training steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'calculating...'}")
print(f"Evaluation every: {training_args.eval_steps} steps")
print(f"Checkpoint every: {training_args.save_steps} steps")



import time

print("STARTING FINE-TUNING!")
print("Estimated time: 4-6 hours")
print("Checkpoints: Every 100 steps to Google Drive\n")

start_time = time.time()

# Train (has built-in progress bar)
trainer.train(resume_from_checkpoint=latest_checkpoint)

# Done
elapsed = (time.time() - start_time) / 3600
print(f"\nTRAINING COMPLETE in {elapsed:.2f} hours")

# Save
print("Saving model...")
final_model_path = f"{DRIVE_PATH}/final_model"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Saved to: {final_model_path}")



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments configured

Key Settings:
Effective batch size: 16
Total epochs: 3
Learning rate: 0.0002
Warmup steps: ~24


Truncating train dataset:   0%|          | 0/4390 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1098 [00:00<?, ? examples/s]

Trainer initialized!

Training will begin with:
Total training steps: 0
Evaluation every: 100 steps
Checkpoint every: 100 steps
STARTING FINE-TUNING!
Estimated time: 4-6 hours
Checkpoints: Every 100 steps to Google Drive



Step,Training Loss,Validation Loss
800,0.428502,0.605884



TRAINING COMPLETE in 0.93 hours
Saving model...
Saved to: /content/drive/MyDrive/CUAD_Project/final_model


In [ ]:
# ========================================
# CELL 21: Visualize Training Progress
# ========================================

import matplotlib.pyplot as plt

# Extract metrics from trainer
train_loss = [x['loss'] for x in trainer.state.log_history if 'loss' in x]
eval_loss = [x['eval_loss'] for x in trainer.state.log_history if 'eval_loss' in x]

steps_train = [x['step'] for x in trainer.state.log_history if 'loss' in x]
steps_eval = [x['step'] for x in trainer.state.log_history if 'eval_loss' in x]

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(steps_train, train_loss, label='Train Loss', alpha=0.7)
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(steps_eval, eval_loss, label='Validation Loss', color='orange')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training curves saved to Google Drive")

# Print final metrics
print(f"\n📊 Final Metrics:")
print(f"   Final train loss: {train_loss[-1]:.4f}")
print(f"   Final val loss: {eval_loss[-1]:.4f}")
print(f"   Best val loss: {min(eval_loss):.4f}")

In [3]:
!hf auth login



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: N
Token is valid (permission: fineGrained).
The token `llama 3.3 70b label generation` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current acti

In [4]:
#Load the model for inference:

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

print("Loading fine-tuned model...")

#Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

#load fine-tuned model
finetuned_model = PeftModel.from_pretrained(base_model, f"{DRIVE_PATH}/final_model", torch_dtype=torch.float16)

#Merge and unload solders the adapters' weights to the base weights, change is irreversible, reduces latency because the matrix multiplication required is reduced.
#Should be done only when all experimentation is done, as we cant separate the base model from adapters
#tricky to do when using 4 bit quantization, cos base weights are stored in float16 and soldering will lead to info loss.
#need to de-quantize before merging. NOTE: unload part makes sure all the adapter metadata are cleared from memory and we have only the new model infused with the adapters left.
#skipping for now...

#finetuned_model = finetuned_model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_PATH}/final_model", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print("Fine-tuned model loaded!")
print(f"Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading fine-tuned model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Fine-tuned model loaded!
Memory: 6.52 GB


In [5]:
#Inference Function:

def analyze_clause(clause_text, contract_type="Service Agreement", jurisdiction="Delaware", renewal_term = "Not Applicable", notice_period = "none",key_provisions = [],
                   target_clause =""):


    instruction = f"""Analyze the following contract clause and provide:
1. Clause Type Classification
2. Risk assessment (HIGH/MEDIUM/LOW) with explanation
3. Plain English summary


**Contract Context:**
Type: {contract_type}
Jurisdiction: {jurisdiction}

**Clause Text:**
{clause_text}

Provide your analysis:"""

    instruction = f"""
  Analyse the following clause and provide:
1. Clause type classification
2. Risk assessment (HIGH/MEDIUM/LOW) with explanation
3. Plain English summary

**Contract Context:**
    Contract Type: {contract_type}
Jurisdiction : {jurisdiction}
Renewal Term: {renewal_term}
Notice Period to terminate renewal: {notice_period}
Key Provisions: {', '.join(key_provisions)}

    **Target Clause Type:** {target_clause}
    **Clause Text:**
      {clause_text}

Provide your analysis:
"""

  #tokenize the inference input
    inputs = tokenizer(instruction, return_tensors = 'pt', truncation = True, max_length = 2048).to(finetuned_model.device)

  #generate
    with torch.no_grad():

      outputs = finetuned_model.generate(**inputs, max_new_tokens = 350, temperature = 0.3, do_sample=True,top_p=0.9, pad_token_id=tokenizer.eos_token_id,
                                         eos_token_id=tokenizer.eos_token_id,)

      full_output = tokenizer.decode(outputs[0], skip_special_tokens = True)


      response = full_output[len(instruction):].strip()

      return response


print("Inference function ready!")




Inference function ready!


In [6]:
#testing the inference function

test_clauses = [
    {   "text":"the franchisee will not directly or indirectly, at any time during the term of this agreement or thereafter, do or cause to be done any act or thing disputing, attacking or in any way impairing the validity of and bkc's right, title or interest in the burger king marks and the burger king system.",
        "type": "Franchise Agreement",
        "jurisdiction": "Florida",
        "clause_type": 'Non-Disparagement',
        "key_provisions": ["Change of Control", "Cap on Liability", "Minimum Commitment"]
    },
    {
        "text": "during the term of this agreement, and for a period of two (2) years thereafter, aucta shall not research, develop, manufacture, file, sell, market, or distribute more than two products containing the active ingredient lamotrigine; nor will aucta directly or indirectly assist any other person or entity in carrying or any such activities.",
        "type": "Exclusive License And Product Devevlopment Agreement",
        "jurisdiction": "Delaware",
        "clause_type": ' Non-Compete',
        "key_provisions": ['Termination for Convenience','Change of Control','Cap on Liability','Minimum Commitment']
    },
    {
        "text": "march 13, 1996",
        "type": "Agreement Date",
        "jurisdiction": "British Columbia,Canada"
    },
    {
        "text" :"franchisee shall not challenge, directly or indirectly, franchisor's interest in, or the validity of, any franchisor property, or any application for registration or trademark registration thereof or any rights of franchisor therein.",
        "type" : "Master Franchise Agreement",
        "jurisdiction": "New York",
        "clause_type": "Covenant Not to Sue",
        "key_provisions": ["Change of Control", "Cap on Liability", "Minimum Commitment"]
    },
    {
       'text': "its products are warranted free from defects of material or workmanship for 3 years after shipment from the manufacturer. equipment purchased from its, which becomes defective within that time period will be repaired by its at its headquarters in san antonio, texas at no cost to comware beyond cost of shipping the equipment to its.",
        'type': "Distributor Agreement",
        'jurisdiction' : "Texas",
        'renewal_term' : '6 months',
        'key_provisions' : ['Termination for Convenience','Change of Control','Cap on Liability','Minimum Commitment'],
        'target_clause' : 'Warranty Duration'
       },
    {
        'text': 'for clarity, a competitive transaction shall not include an agreement for use, integration or interfacing, or co-marketing, of the ehave companion solution with other services, solutions, devices, goods or products, where such other services, solutions, devices, goods or products do not contain the same or similar functionality of the ehave companion solution, but provides for a complementary solution.',
        'jurisdiction' : 'Ontario, Canada',
        'type': 'License and Reseller Agreement',
        'target_clause' : 'Competitive Restriction Exception',
        'key_provisions' : ['Termination for Convenience','Change of Control','Cap on Liability','Minimum Commitment']
    }
]


print("Testing fine-tuned model on 3 samples...\n")

for i, sample in enumerate(test_clauses):

  print("="*80)
  print(f"\nSample {i+1}:")
  print(f"Contract Type: {sample['type']}")
  print(f"Jurisdiction: {sample['jurisdiction']}")
  print(f"Clause Text: {sample['text']}")
  print("\nAnalysis:")
  print(analyze_clause(sample['text'], sample['type'], sample['jurisdiction']))
  print("="*80)


Testing fine-tuned model on 3 samples...


Sample 1:
Contract Type: Franchise Agreement
Jurisdiction: Florida
Clause Text: the franchisee will not directly or indirectly, at any time during the term of this agreement or thereafter, do or cause to be done any act or thing disputing, attacking or in any way impairing the validity of and bkc's right, title or interest in the burger king marks and the burger king system.

Analysis:
**Risk Assessment:** MEDIUM
**Explanation:** The clause restricts the franchisee's ability to dispute or impair BKC's rights to the Burger King marks and system, which could limit the franchisee's ability to defend themselves in a dispute. However, the risk is mitigated by the fact that the clause only applies to actions taken during or after the term of the agreement. The jurisdiction of Florida may also impact the enforceability of this clause, but the overall risk remains moderate.

**Plain English Summary:** The franchisee is not allowed to challenge BKC's o

In [7]:
!pip install -q gradio

In [8]:
# ========================================
# GRADIO: Complete UI with Your Samples
# ========================================

import gradio as gr

def analyze_clause_wrapper(clause_text, target_clause, contract_type, jurisdiction,
                          renewal_term, notice_period,
                          has_liability_cap, has_termination, has_change_control, has_minimum_commitment):
    """
    Wrapper to convert UI inputs to your function signature
    """

    if not clause_text.strip():
        return "⚠️ Please enter clause text"

    if target_clause == "Select clause type...":
        return "⚠️ Please select clause type"

    # Build key provisions list
    key_provisions = []
    if has_liability_cap:
        key_provisions.append("Cap on Liability")
    if has_termination:
        key_provisions.append("Termination for Convenience")
    if has_change_control:
        key_provisions.append("Change of Control")
    if has_minimum_commitment:
        key_provisions.append("Minimum Commitment")

    # Call your actual function
    response = analyze_clause(
        clause_text=clause_text,
        contract_type=contract_type,
        jurisdiction=jurisdiction,
        renewal_term=renewal_term,
        notice_period=notice_period,
        key_provisions=key_provisions,
        target_clause=target_clause
    )

    return response


with gr.Blocks(title="Contract Risk Analyzer", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # ⚖️ Contract Risk Analyzer
    ### Context-Aware Legal Clause Analysis

    Fine-tuned Llama 3.2 3B (QLoRA) provides:
    - 🎯 Risk assessment with jurisdiction-aware reasoning
    - 📝 Plain English explanations
    - 🔍 Contract context integration
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📝 Input")

            clause_input = gr.Textbox(
                label="Contract Clause Text",
                placeholder="Paste your contract clause here...",
                lines=6
            )

            target_clause_input = gr.Dropdown(
                choices=[
                    "Select clause type...",
                    "Non-Compete",
                    "Non-Disparagement",
                    "Covenant Not to Sue",
                    "Exclusivity",
                    "License Grant",
                    "Cap on Liability",
                    "Termination for Convenience",
                    "Change of Control",
                    "Minimum Commitment",
                    "Volume Restriction",
                    "Anti-Assignment",
                    "Post-Termination Services",
                    "Audit Rights",
                    "Insurance",
                    "Renewal Term",
                    "Notice to Terminate Renewal",
                    "IP Ownership Assignment",
                    "No-Solicit of Employees",
                    "Warranty Duration",
                    "Competitive Restriction Exception",
                ],
                value="Select clause type...",
                label="Clause Type",
                allow_custom_value=True
            )

            gr.Markdown("### Contract Context")

            with gr.Row():
                contract_type_input = gr.Dropdown(
                    choices=[
                        "Service Agreement",
                        "Employment Agreement",
                        "License Agreement",
                        "Distribution Agreement",
                        "Distributor Agreement",
                        "Franchise Agreement",
                        "Master Franchise Agreement",
                        "License and Reseller Agreement",
                        "Exclusive License And Product Development Agreement",
                        "NDA",
                        "Other"
                    ],
                    value="Service Agreement",
                    label="Contract Type",
                    allow_custom_value=True
                )

                jurisdiction_input = gr.Dropdown(
                    choices=[
                        "Delaware",
                        "California",
                        "New York",
                        "Texas",
                        "Florida",
                        "Illinois",
                        "Washington",
                        "Ontario, Canada",
                        "British Columbia, Canada",
                        "Other"
                    ],
                    value="Delaware",
                    label="Jurisdiction",
                    allow_custom_value=True
                )

            with gr.Row():
                renewal_term_input = gr.Textbox(
                    label="Renewal Term",
                    placeholder="e.g., 3 years, 6 months, Perpetual",
                    value="Not Applicable"
                )

                notice_period_input = gr.Textbox(
                    label="Notice Period to Terminate",
                    placeholder="e.g., 30 days, 60 days, none",
                    value="none"
                )

            gr.Markdown("### Key Provisions Present")

            with gr.Row():
                has_liability_cap = gr.Checkbox(label="Cap on Liability", value=False)
                has_termination = gr.Checkbox(label="Termination for Convenience", value=False)

            with gr.Row():
                has_change_control = gr.Checkbox(label="Change of Control", value=False)
                has_minimum_commitment = gr.Checkbox(label="Minimum Commitment", value=False)

            analyze_btn = gr.Button("🔍 Analyze Clause", variant="primary", size="lg")

        with gr.Column(scale=1):
            gr.Markdown("## 📊 Analysis")

            output = gr.Textbox(
                label="Model Output",
                lines=16,
                interactive=False,
                show_copy_button=True
            )

    gr.Markdown("## 💡 Real CUAD Dataset Examples")

    gr.Examples(
        examples=[
            [
                # Example 1: Non-Disparagement
                "the franchisee will not directly or indirectly, at any time during the term of this agreement or thereafter, do or cause to be done any act or thing disputing, attacking or in any way impairing the validity of and bkc's right, title or interest in the burger king marks and the burger king system.",
                "Non-Disparagement",
                "Franchise Agreement",
                "Florida",
                "Not Applicable",
                "none",
                True,   # Cap on Liability
                False,  # Termination for Convenience
                True,   # Change of Control
                True    # Minimum Commitment
            ],
            [
                # Example 2: Non-Compete
                "during the term of this agreement, and for a period of two (2) years thereafter, aucta shall not research, develop, manufacture, file, sell, market, or distribute more than two products containing the active ingredient lamotrigine; nor will aucta directly or indirectly assist any other person or entity in carrying or any such activities.",
                "Non-Compete",
                "Exclusive License And Product Development Agreement",
                "Delaware",
                "Not Applicable",
                "none",
                True,   # Cap on Liability
                True,   # Termination for Convenience
                True,   # Change of Control
                True    # Minimum Commitment
            ],
            [
                # Example 3: Covenant Not to Sue
                "franchisee shall not challenge, directly or indirectly, franchisor's interest in, or the validity of, any franchisor property, or any application for registration or trademark registration thereof or any rights of franchisor therein.",
                "Covenant Not to Sue",
                "Master Franchise Agreement",
                "New York",
                "Not Applicable",
                "none",
                True,   # Cap on Liability
                False,  # Termination for Convenience
                True,   # Change of Control
                True    # Minimum Commitment
            ],
            [
                # Example 4: Warranty Duration
                "its products are warranted free from defects of material or workmanship for 3 years after shipment from the manufacturer. equipment purchased from its, which becomes defective within that time period will be repaired by its at its headquarters in san antonio, texas at no cost to comware beyond cost of shipping the equipment to its.",
                "Warranty Duration",
                "Distributor Agreement",
                "Texas",
                "6 months",
                "none",
                True,   # Cap on Liability
                True,   # Termination for Convenience
                True,   # Change of Control
                True    # Minimum Commitment
            ],
            [
                # Example 5: Competitive Restriction Exception
                "for clarity, a competitive transaction shall not include an agreement for use, integration or interfacing, or co-marketing, of the ehave companion solution with other services, solutions, devices, goods or products, where such other services, solutions, devices, goods or products do not contain the same or similar functionality of the ehave companion solution, but provides for a complementary solution.",
                "Competitive Restriction Exception",
                "License and Reseller Agreement",
                "Ontario, Canada",
                "Not Applicable",
                "none",
                True,   # Cap on Liability
                True,   # Termination for Convenience
                True,   # Change of Control
                True    # Minimum Commitment
            ],
        ],
        inputs=[
            clause_input,
            target_clause_input,
            contract_type_input,
            jurisdiction_input,
            renewal_term_input,
            notice_period_input,
            has_liability_cap,
            has_termination,
            has_change_control,
            has_minimum_commitment
        ],
        label="Click any example to load it"
    )

    # Connect button
    analyze_btn.click(
        fn=analyze_clause_wrapper,
        inputs=[
            clause_input,
            target_clause_input,
            contract_type_input,
            jurisdiction_input,
            renewal_term_input,
            notice_period_input,
            has_liability_cap,
            has_termination,
            has_change_control,
            has_minimum_commitment
        ],
        outputs=output
    )

    gr.Markdown("""
    ---
    ### 📚 Project Details

    **Model:**
    - Base: Llama 3.2 3B Instruct
    - Fine-tuning: QLoRA (LoRA rank 16, alpha 32, ~20M trainable params)
    - Training: 3 epochs, 825 steps on 3,900 samples
    - Dataset: CUAD (Contract Understanding Atticus Dataset)

    **Key Learnings:**
    - Model performs best with rich contract context matching training distribution
    - Token-level cross-entropy loss prioritizes explanation fluency over discrete classification
    - Risk assessment quality benefits from jurisdiction and multi-provision context
    - Train/inference distribution alignment is critical for generative task performance

    **Evaluation:**
    - Risk classification: ~55-60% (vs 63% majority baseline)
    - Explanation quality: High (context-aware, legally sound reasoning)
    - Summarization: High (captures key points in plain language)
    """)

print("✅ Gradio interface ready!")

/tmp/ipython-input-1033/1984641036.py:45: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Contract Risk Analyzer", theme=gr.themes.Soft()) as demo:


✅ Gradio interface ready!


In [9]:
demo.launch(share = True, debug = True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bd0a65a483ae875344.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bd0a65a483ae875344.gradio.live


In [10]:
!gradio deploy

Need 'write' access token to create a Spaces repo.
Creating new Spaces Repo in '/content'. Collecting metadata, press Enter to 
accept default value.
Enter Spaces app title [content]: Legal Contract Risk analyzer and summarizer
Formatted to Legal_Contract_Risk_analyzer_and_summarizer. 
Enter Gradio app file : app.py
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /usr/local/lib/python3.12/dist-packages/gradio/cli/commands/deploy_space.py: │
│ 301 in deploy                                                                │
│                                                                              │
│   298 │   │   print(                                                         │
│   299 │   │   │   f"Creating new Spaces Repo in '{repo_directory}'. Collecti │
│   300 │   │   )                                                              │
│ ❱ 301 │   │   configuration = add_configuration_to_readme(                   │
│   302 │   │   │   title,        

In [16]:
#uploading model to hf hub

# ========================================
# Upload Fine-Tuned Model to HF Hub
# ========================================

from huggingface_hub import HfApi, create_repo, login

# Login to HF
login()

# Create model repo
repo_name = "contract-analyzer-and-summarizer-Qlora"
username = "bharathgrinds"

create_repo(
    repo_id=f"{username}/{repo_name}",
    exist_ok=True,
    repo_type="model"
)

# Upload model files
print("Uploading model...")

# Your fine-tuned model path
model_path = f"{DRIVE_PATH}/final_model"

# Upload
api = HfApi()
api.upload_folder(
    folder_path=model_path,
    repo_id=f"{username}/{repo_name}",
    repo_type="model"
)

print(f"Model uploaded to: https://huggingface.co/{username}/{repo_name}")

Uploading model...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

  ...inal_model/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...l_model/training_args.bin:   2%|1         |   107B / 5.78kB            

Model uploaded to: https://huggingface.co/bharathgrinds/contract-analyzer-and-summarizer-Qlora


In [15]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: N
Token is valid (permission: write).
The token `legal_contract_analyzer_model_upload` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current acti